# Silver Layer - Stores

Transform raw Bronze data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.stores`  
**Target:** `end-to-end_pipeline.silver.stores`

**Approach:** Profile → Inspect → Transform → Validate

## Step 1: Profile Bronze Data

**Inspect data quality issues before transformation:**

* Duplicate store_ids
* NULL values in key fields (store_id, store_name, city, region, manager_name)
* Inconsistent store_type values
* Inconsistent region values

This single query checks all quality dimensions.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT store_id) AS distinct_store_ids,
    COUNT(*) - COUNT(DISTINCT store_id) AS duplicate_stores,

    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS null_store_ids,
    SUM(CASE WHEN store_name IS NULL THEN 1 ELSE 0 END) AS null_store_names,
    SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS null_cities,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_regions,
    SUM(CASE WHEN manager_name IS NULL THEN 1 ELSE 0 END) AS null_managers,

    COUNT(DISTINCT store_type) AS store_type_variations,
    COUNT(DISTINCT region) AS region_variations

FROM `end-to-end_pipeline`.bronze.stores;

total_rows,distinct_store_ids,duplicate_stores,null_store_ids,null_store_names,null_cities,null_regions,null_managers,store_type_variations,region_variations
30,30,0,0,0,0,0,1,4,6


## Step 2: Inspect Categorical Values

**Review actual store_type and region values to identify standardization needs:**

* Store type variations (online, E-Commerce → Online)
* Region casing and whitespace (" north " → North)

This inspection guides the explicit CASE logic in the transformation.

In [0]:
%sql

SELECT
    store_type,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.stores
GROUP BY store_type
ORDER BY records DESC;

store_type,records
Physical Store,21
Online,7
online,1
E-Commerce,1


In [0]:
%sql

SELECT
    region,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.stores
GROUP BY region
ORDER BY records DESC;

region,records
South,9
North,8
East,4
Central,4
West,4
north,1


## Step 3: Transform to Silver

**Apply all data quality fixes in one pass:**

**Data Cleaning:**
* TRIM whitespace from all identifiers and text (handles " north " → "north")
* Standardize store names, city, country with INITCAP
* Convert NULL manager_name → 'Unknown'

**Categorical Standardization:**
* Standardize store_type: online, E-Commerce → Online
* Standardize region casing: north → North

**Data Type Enforcement:**
* Convert opening_date to DATE format with TRY_TO_DATE

This creates a clean, analytics-ready Silver table.


In [0]:
%sql

-- ============================================================
-- CELL 3: TRANSFORM BRONZE → SILVER STORES
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.stores AS

SELECT
    TRIM(store_id) AS store_id,
    TRIM(store_name) AS store_name,

    -- Standardize store type: online, E-Commerce → Online
    CASE
        WHEN LOWER(TRIM(store_type)) = 'online' THEN 'Online'
        WHEN LOWER(TRIM(store_type)) = 'e-commerce' THEN 'Online'
        WHEN LOWER(TRIM(store_type)) = 'physical store' THEN 'Physical Store'
        ELSE INITCAP(TRIM(store_type))
    END AS store_type,

    INITCAP(TRIM(city)) AS city,

    -- Standardize region: explicit handling of whitespace and casing
    CASE
        WHEN LOWER(TRIM(region)) = 'north' THEN 'North'
        WHEN LOWER(TRIM(region)) = 'south' THEN 'South'
        WHEN LOWER(TRIM(region)) = 'east' THEN 'East'
        WHEN LOWER(TRIM(region)) = 'west' THEN 'West'
        WHEN LOWER(TRIM(region)) = 'central' THEN 'Central'
        ELSE INITCAP(TRIM(region))
    END AS region,

    INITCAP(TRIM(country)) AS country,

    TRY_TO_DATE(opening_date, 'yyyy-MM-dd') AS opening_date,

    -- Preserve store with missing manager
    COALESCE(INITCAP(TRIM(manager_name)), 'Unknown') AS manager_name

FROM `end-to-end_pipeline`.bronze.stores

WHERE store_id IS NOT NULL;

num_affected_rows,num_inserted_rows


## Step 4: Validate Silver Data

**Verify all transformations were successful.**

**Expected Results:**
* total_rows = 30
* distinct_store_ids = 30
* remaining_duplicates = 0
* null_store_ids = 0
* null_managers = 0 (all converted to 'Unknown')
* invalid_opening_dates = 0
* invalid_store_types = 0 (only Physical Store/Online)
* invalid_regions = 0 (only North/South/East/West/Central)
* validation_status = 'PASS'

If any metric is unexpected, the transformation has an issue.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE SILVER STORES
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT store_id) AS distinct_store_ids,
        COUNT(*) - COUNT(DISTINCT store_id) AS remaining_duplicates,

        SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END)
            AS null_store_ids,

        SUM(CASE WHEN manager_name IS NULL THEN 1 ELSE 0 END)
            AS null_managers,

        SUM(CASE WHEN opening_date IS NULL THEN 1 ELSE 0 END)
            AS invalid_opening_dates,

        SUM(
            CASE
                WHEN store_type IN ('Physical Store', 'Online')
                    THEN 0
                ELSE 1
            END
        ) AS invalid_store_types,

        SUM(
            CASE
                WHEN region IN (
                    'North',
                    'South',
                    'East',
                    'West',
                    'Central'
                )
                    THEN 0
                ELSE 1
            END
        ) AS invalid_regions

    FROM `end-to-end_pipeline`.silver.stores
)

SELECT
    *,

    CASE
        WHEN total_rows = 30
            AND distinct_store_ids = 30
            AND remaining_duplicates = 0
            AND null_store_ids = 0
            AND null_managers = 0
            AND invalid_opening_dates = 0
            AND invalid_store_types = 0
            AND invalid_regions = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation;

total_rows,distinct_store_ids,remaining_duplicates,null_store_ids,null_managers,invalid_opening_dates,invalid_store_types,invalid_regions,validation_status
30,30,0,0,0,0,0,0,PASS
